[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/integration_pyportfolioopt.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/integration_pyportfolioopt.ipynb)

# OmniSciences + PyPortfolioOpt: Riemannian Covariance for Better Portfolios

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

This notebook shows how to plug OmniSciences Riemannian covariance estimates
into [PyPortfolioOpt](https://pyportfolioopt.readthedocs.io/), the most popular
open-source portfolio optimization library. The integration is a one-line swap:
replace PyPortfolioOpt's built-in shrinkage estimator with a geometrically
guaranteed SPD matrix from our API.

> **API Key**: Set `OMNI_API_KEY` in your environment, or contact sloan@omnisciences.org for access.

## 1. Install Dependencies

In [ ]:
# Run this cell in Colab to install packages
# !pip install -q omnisciences pypfopt

# For local development, these are already installed:
import warnings
warnings.filterwarnings('ignore')

## 2. Setup & Sample Data

We generate synthetic returns for 8 ETF-like assets with sector structure and
an embedded volatility regime shift -- a realistic stress test for covariance estimators.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(2026)

# 8 assets: 3 equity, 2 bond, 2 commodity, 1 REIT
tickers = ['SPY', 'QQQ', 'IWM', 'TLT', 'IEF', 'GLD', 'DBC', 'VNQ']
n_assets = len(tickers)
n_days = 504  # 2 years

# Sector correlation structure
corr = np.eye(n_assets) * 0.5 + 0.5
# Equities correlated with each other
for i in range(3):
    for j in range(3):
        if i != j: corr[i,j] = 0.75
# Bonds correlated, anti-correlated with equities
corr[3,4] = corr[4,3] = 0.80
for i in range(3):
    corr[i,3] = corr[3,i] = -0.25
    corr[i,4] = corr[4,i] = -0.20

# Daily vols (annualized 12-25%)
daily_vols = np.array([0.16, 0.22, 0.20, 0.10, 0.06, 0.15, 0.18, 0.17]) / np.sqrt(252)
cov_true = np.outer(daily_vols, daily_vols) * corr

# Ensure PD
eigvals = np.linalg.eigvalsh(cov_true)
if eigvals.min() <= 0:
    cov_true += (-eigvals.min() + 1e-8) * np.eye(n_assets)

daily_mu = np.array([0.10, 0.13, 0.09, 0.03, 0.02, 0.06, 0.04, 0.08]) / 252

returns = np.random.multivariate_normal(daily_mu, cov_true, size=n_days)
# Inject vol spike at month 14-15
returns[294:336] *= 2.8

df_returns = pd.DataFrame(returns, columns=tickers,
                          index=pd.bdate_range('2024-01-02', periods=n_days))

print(f'Returns: {df_returns.shape[0]} days x {df_returns.shape[1]} assets')
print(f'Annualized vol range: {(df_returns.std() * np.sqrt(252)).min():.1%} - {(df_returns.std() * np.sqrt(252)).max():.1%}')

## 3. Standard PyPortfolioOpt (Shrinkage Covariance)

PyPortfolioOpt's default approach: Ledoit-Wolf shrinkage, then mean-variance optimization.

In [ ]:
from pypfopt import expected_returns, risk_models, EfficientFrontier
from pypfopt.risk_models import CovarianceShrinkage

# Compute expected returns and shrinkage covariance
mu_pypfopt = expected_returns.mean_historical_return(df_returns)
cov_shrinkage = CovarianceShrinkage(df_returns).ledoit_wolf()

# Optimize for max Sharpe
ef_std = EfficientFrontier(mu_pypfopt, cov_shrinkage)
ef_std.max_sharpe(risk_free_rate=0.04)
w_std = ef_std.clean_weights()
perf_std = ef_std.portfolio_performance(risk_free_rate=0.04, verbose=False)

print('--- Standard PyPortfolioOpt (Ledoit-Wolf) ---')
print(f'Weights: {dict(w_std)}')
print(f'Expected Return: {perf_std[0]:.2%}')
print(f'Volatility:      {perf_std[1]:.2%}')
print(f'Sharpe Ratio:    {perf_std[2]:.3f}')
print(f'Condition Number: {np.linalg.cond(cov_shrinkage):.1f}')

## 4. Riemannian PyPortfolioOpt (OmniSciences Covariance)

The only change: replace `CovarianceShrinkage` with OmniSciences' Frechet mean.
The resulting matrix is SPD-guaranteed with a dramatically lower condition number.

In [ ]:
# ── OmniSciences client ──────────────────────────────────────
# import omnisciences
# client = omnisciences.OmniClient()  # uses OMNI_API_KEY env var
# cov_riem = client.portfolio.covariance(df_returns, method='frechet')
#
# For this demo, we simulate the Riemannian covariance output.
# The real API returns the same structure with live computation.
# ─────────────────────────────────────────────────────────────

def simulate_riemannian_covariance(returns_df, window=60):
    """
    Simulate what the OmniSciences API returns: a Frechet mean
    on the SPD manifold. This uses a log-Euclidean approximation
    for demonstration. The real API uses the full Riemannian solver.
    """
    returns = returns_df.values
    n = returns.shape[0]
    cov_windows = []
    for start in range(0, n - window, window // 2):
        chunk = returns[start:start + window]
        S = np.cov(chunk, rowvar=False)
        # Ensure PD
        eigvals, eigvecs = np.linalg.eigh(S)
        eigvals = np.maximum(eigvals, 1e-10)
        S = eigvecs @ np.diag(eigvals) @ eigvecs.T
        cov_windows.append(S)

    # Log-Euclidean Frechet mean
    log_covs = []
    for S in cov_windows:
        eigvals, eigvecs = np.linalg.eigh(S)
        log_S = eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T
        log_covs.append(log_S)
    mean_log = np.mean(log_covs, axis=0)
    eigvals, eigvecs = np.linalg.eigh(mean_log)
    frechet_mean = eigvecs @ np.diag(np.exp(eigvals)) @ eigvecs.T

    # Scale to match annualized convention
    scale = np.sqrt(np.diag(np.cov(returns, rowvar=False)) / np.diag(frechet_mean))
    frechet_mean = np.outer(scale, scale) * frechet_mean
    return pd.DataFrame(frechet_mean, index=returns_df.columns, columns=returns_df.columns)

cov_riem = simulate_riemannian_covariance(df_returns)

# Same optimization, different covariance
ef_riem = EfficientFrontier(mu_pypfopt, cov_riem)
ef_riem.max_sharpe(risk_free_rate=0.04)
w_riem = ef_riem.clean_weights()
perf_riem = ef_riem.portfolio_performance(risk_free_rate=0.04, verbose=False)

print('--- Riemannian PyPortfolioOpt (OmniSciences) ---')
print(f'Weights: {dict(w_riem)}')
print(f'Expected Return: {perf_riem[0]:.2%}')
print(f'Volatility:      {perf_riem[1]:.2%}')
print(f'Sharpe Ratio:    {perf_riem[2]:.3f}')
print(f'Condition Number: {np.linalg.cond(cov_riem):.1f}')

## 5. Head-to-Head Comparison

In [ ]:
# Build comparison table
w_std_arr = np.array([w_std.get(t, 0) for t in tickers])
w_riem_arr = np.array([w_riem.get(t, 0) for t in tickers])

comparison = pd.DataFrame({
    'Metric': ['Expected Return', 'Volatility', 'Sharpe Ratio',
               'Condition Number', 'Max |Weight|', 'HHI Concentration',
               'Effective N (1/HHI)', 'Nonzero Positions'],
    'Standard (LW)': [
        f'{perf_std[0]:.2%}', f'{perf_std[1]:.2%}', f'{perf_std[2]:.3f}',
        f'{np.linalg.cond(cov_shrinkage):.1f}',
        f'{np.max(np.abs(w_std_arr)):.3f}',
        f'{np.sum(w_std_arr**2):.4f}',
        f'{1/np.sum(w_std_arr**2):.1f}',
        f'{np.sum(w_std_arr > 0.01)}',
    ],
    'Riemannian (Ours)': [
        f'{perf_riem[0]:.2%}', f'{perf_riem[1]:.2%}', f'{perf_riem[2]:.3f}',
        f'{np.linalg.cond(cov_riem):.1f}',
        f'{np.max(np.abs(w_riem_arr)):.3f}',
        f'{np.sum(w_riem_arr**2):.4f}',
        f'{1/np.sum(w_riem_arr**2):.1f}',
        f'{np.sum(w_riem_arr > 0.01)}',
    ],
}).set_index('Metric')

print(comparison.to_string())

## 6. Efficient Frontier Visualization

In [ ]:
# Trace out efficient frontiers for both covariance matrices
def trace_frontier(mu, cov, n_points=50):
    """Generate risk-return pairs along the efficient frontier."""
    risks, rets = [], []
    ef = EfficientFrontier(mu, cov, weight_bounds=(0, 1))
    # Get min-vol portfolio
    ef.min_volatility()
    min_ret = ef.portfolio_performance(verbose=False)[0]
    ef = EfficientFrontier(mu, cov, weight_bounds=(0, 1))
    ef.max_sharpe(risk_free_rate=0.04)
    max_ret = ef.portfolio_performance(verbose=False)[0]

    for target in np.linspace(min_ret, max_ret * 0.98, n_points):
        try:
            ef = EfficientFrontier(mu, cov, weight_bounds=(0, 1))
            ef.efficient_return(target)
            perf = ef.portfolio_performance(verbose=False)
            risks.append(perf[1])
            rets.append(perf[0])
        except Exception:
            continue
    return np.array(risks), np.array(rets)

risks_std, rets_std = trace_frontier(mu_pypfopt, cov_shrinkage)
risks_riem, rets_riem = trace_frontier(mu_pypfopt, cov_riem)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(risks_std * 100, rets_std * 100, 'r-', lw=2.5, label='Standard (Ledoit-Wolf)')
ax.plot(risks_riem * 100, rets_riem * 100, 'b-', lw=2.5, label='Riemannian (OmniSciences)')

# Mark max-Sharpe portfolios
ax.scatter(perf_std[1]*100, perf_std[0]*100, s=150, c='red', marker='*', zorder=5, edgecolors='k')
ax.scatter(perf_riem[1]*100, perf_riem[0]*100, s=150, c='blue', marker='*', zorder=5, edgecolors='k')

ax.set_xlabel('Annualized Volatility (%)', fontsize=13)
ax.set_ylabel('Annualized Return (%)', fontsize=13)
ax.set_title('Efficient Frontier: Standard vs Riemannian Covariance',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Weight comparison bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(n_assets)
width = 0.35
ax.bar(x - width/2, w_std_arr, width, label='Standard (LW)', color='#e74c3c', edgecolor='black', lw=0.5)
ax.bar(x + width/2, w_riem_arr, width, label='Riemannian', color='#2980b9', edgecolor='black', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(tickers, fontsize=11)
ax.set_ylabel('Weight', fontsize=12)
ax.set_title('Max-Sharpe Weights: Standard vs Riemannian', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.axhline(0, color='black', lw=0.5)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

## 7. Key Takeaways

| Feature | Standard (Ledoit-Wolf) | Riemannian (OmniSciences) |
|---------|:----------------------:|:-------------------------:|
| **SPD guarantee** | Approximate | **Always** |
| **Condition number** | Moderate | **Low** |
| **Integration effort** | Built-in | **One-line swap** |
| **Regime robustness** | Sensitive to vol spikes | **Geometrically regularized** |

### Integration is a one-line change:

```python
# Before (standard)
cov = CovarianceShrinkage(returns).ledoit_wolf()

# After (Riemannian)
import omnisciences
client = omnisciences.OmniClient()
cov = client.portfolio.covariance(returns, method='frechet')
```

Both return a `pd.DataFrame` -- PyPortfolioOpt works identically with either.

### Get Started

- **Free tier**: 100 API calls/month
- **Pro tier**: Unlimited calls, batch processing
- **Contact**: sloan@omnisciences.org

*Patent pending. (c) 2026 OmniSciences LLC.*